# Filtrado colaborativ ítem - ítem basado en KNN

El filtrado colaborativo ítem-ítem basado en KNN es una técnica de recomendación que sugiere a un usuario elementos similares a los que ya ha evaluado positivamente en el pasado. En lugar de buscar usuario con gustos parecidos, analiza las coincidencias en los patrones de calificación que recibe cada objeto por parte de toda la comunidad.

Se toma un muestra de 10.000 ítems (populares) para reducir el uso de memoria y garantizar que el entrenamiento de K-NN sea computacionalmente viable en este entorno.

In [1]:
import pandas as pd

In [2]:
ratings_df = pd.read_csv('ratings_limpios.csv')
resumen_usuario = pd.read_csv('resumen_usuario.csv')

In [3]:
items_frecuentes = (
    ratings_df["ISBN"]
    .value_counts()
    .head(10000)
    .index
    )

ratings_item = ratings_df[
    ratings_df["ISBN"].isin(items_frecuentes)
    ].copy()


In [41]:
from surprise import Dataset, Reader, KNNWithMeans
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(1,10))
surprise_df = ratings_item.copy()
surprise_df.columns = ["uid", "iid", "rating"]

data = Dataset.load_from_df(surprise_df, reader)

# 0.1, 0.2, 0.3
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# 20, 40, 60
# cosine, pearson, msd
algo = KNNWithMeans(k=40, sim_options={'name': 'msd', 'user_based': False})
algo.fit(trainset)
predictions = algo.test(testset)

Computing the msd similarity matrix...
Done computing similarity matrix.


In [42]:
# Transformación a dataframe
preds_df = pd.DataFrame([
    {
        'User-ID':pred.uid,
        'ISBN': pred.iid,
        'r_ui': pred.r_ui,
        'est': pred.est
    }
    for pred in predictions
])

In [43]:
from metrics import evaluar_metricas_usuario

K = 10
metricas_usuarios = (
    preds_df
    .groupby('User-ID')
    .apply(evaluar_metricas_usuario, k=K)
    .reset_index()
)


df_evaluacion = pd.merge(
    metricas_usuarios,
    resumen_usuario,
    on='User-ID',
    how='inner',
    validate='one_to_one'
)

In [44]:
# A. Por Grupo Etario (Demográfico)
print(f"=== Métricas por Grupo Etario (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Etario')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# B. Por Historial de Interacciones (Comportamiento)
print(f"\n=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Historial')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

# C. Por Grado de Exigencia (Comportamiento)
print(f"\n=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K={K}) ===")
print(df_evaluacion.groupby('Grupo_Exigencia')[['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']].mean())

=== Métricas por Grupo Etario (K=10) ===
                   MAE      CG@10     DCG@10   NDCG@10
Grupo_Etario                                          
Adultos       1.371763  16.822148  12.112145  0.988474
Jóvenes       1.362560  14.228135  11.134798  0.991210
Mayores       1.329511  11.675159   9.725644  0.992630

=== Métricas por Historial de Interacciones (0: Corto, 1: Largo) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Historial                                          
0                1.355434   7.659243   7.648786  0.999951
1                1.395718  17.294394  12.388788  0.986777

=== Métricas por Grado de Exigencia (0: Exigente, 1: Normal, 2: Generoso) (K=10) ===
                      MAE      CG@10     DCG@10   NDCG@10
Grupo_Exigencia                                          
0                2.565244   7.271782   6.194757  0.992425
1                1.214231  15.628031  11.598841  0.989119
2                1.718336  15.498241  12.592179  0.997353

Obserbaciones:

Grupo Etario:

El desempeño es bastante homogéneo. Los mayores tienen el menor MAE y el NDCG más alta, aunque las diferencias son pequeñas.

En cuanto a CG y DCG, los adultos presentan los valores más altos. Sin embargo esto puede deberse a que los grupos tienen distinta cantidad de iteracciones evaluadas y no necesariamente a una mejor calidad de predicción.

Grupo separado por historial:

Los usuario con historial largo presentan un MAE ligeramente mayor que los usuarios con historial corto. Ademas presenta valores superores de CG y DCG lo cual es esperable dado que tiene más libros evaluados.
Lo mismo que se vio en los otros modelo el NDCG en el historial corto es casi perfecto lo cual se tiene que tomar con cautela. 

Grado de exigencia:

Los usuario exigentes presentan nuevamente un MAE mucho mayor, mientras que los normales obtiene el menor error. El patron se mantiene en cada uno de los modelos.

A pesar de esto su NDCG sigue siendo elevado. Esto sugiere que el modelo puede ordenar relativamente bien los ítems, aunque sus predicciones numéricas estén alejadas de las calificaciones reales.




### Hipótesis: los usuarios con un historial de interacciones largo se ven más beneficiados por el sistema de recomendación que aquellos con un historial corto.

In [28]:
from scipy.stats import shapiro

g_corto = df_evaluacion[df_evaluacion['Grupo_Historial'] == 0]
g_largo = df_evaluacion[df_evaluacion['Grupo_Historial'] == 1]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()

    print(f"\n {metrica}")

    stat, p = shapiro(v_corto)
    print(f"Test de Shapiro-Wilk — Historial corto — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_largo)
    print(f"Test de Shapiro-Wilk — Historial largo — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")


 MAE
Test de Shapiro-Wilk — Historial corto — MAE: estadístico=0.842, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — MAE: estadístico=0.899, p-valor=0.000

 CG@10
Test de Shapiro-Wilk — Historial corto — CG@10: estadístico=0.900, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — CG@10: estadístico=0.699, p-valor=0.000

 DCG@10
Test de Shapiro-Wilk — Historial corto — DCG@10: estadístico=0.926, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — DCG@10: estadístico=0.820, p-valor=0.000

 NDCG@10
Test de Shapiro-Wilk — Historial corto — NDCG@10: estadístico=0.012, p-valor=0.000
Test de Shapiro-Wilk — Historial largo — NDCG@10: estadístico=0.529, p-valor=0.000


c:\Users\esper\Desktop\tp-recomendacion\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 6154.
  res = hypotest_fun_out(*samples, **kwds)
c:\Users\esper\Desktop\tp-recomendacion\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 15145.
  res = hypotest_fun_out(*samples, **kwds)


Ninguna de las métricas evaluadas mediante el test de Shapiro-Wilk presentó un p-valor mayor a 0,05, por lo que se rechaza la hipótesis de normalidad para todas las métricas.

In [11]:
import scipy.stats as stats

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()

    stat, p = stats.levene(v_largo,v_corto)
    print(f"Test de Levene para {metrica}: Estadístico={stat:.3f}, p-valor={p:.3f}")

Test de Levene para MAE: Estadístico=75.535, p-valor=0.000
Test de Levene para CG@10: Estadístico=1192.060, p-valor=0.000
Test de Levene para DCG@10: Estadístico=1439.446, p-valor=0.000
Test de Levene para NDCG@10: Estadístico=676.255, p-valor=0.000


Mediante el test de Levene se determinó que las métricas de los grupos separados según la longitud del historial no presentan homocedasticidad.

Dado que no se cumplen los supuestos de normalidad ni de homocedasticidad, se recurre al test no paramétrico de Kruskal-Wallis.

In [12]:
alpha = 0.05

for metrica in metricas:
    # Extracción de valores limpios
    v_corto = g_corto[metrica].dropna()
    v_largo = g_largo[metrica].dropna()
    
    #Test de shapiro-wilk
    stat, p = stats.kruskal(v_largo, v_corto)
    print(f"\nTest de Kruskal-Wallis - {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    if p > alpha:
        print("No hay suficiente evidencia para rechazar la hipótesis nula.")
        print(f"No hay una diferencia significativa en {metrica} entre historiales largos y cortos.")
    else:
        print("Se rechaza la hipótesis nula.")
        print(f"Existe una diferencia significativa en {metrica} entre historiales largos y cortos.")


Test de Kruskal-Wallis - MAE: estadístico=24.243, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en MAE entre historiales largos y cortos.

Test de Kruskal-Wallis - CG@10: estadístico=1573.923, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en CG@10 entre historiales largos y cortos.

Test de Kruskal-Wallis - DCG@10: estadístico=1475.349, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en DCG@10 entre historiales largos y cortos.

Test de Kruskal-Wallis - NDCG@10: estadístico=1219.029, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en NDCG@10 entre historiales largos y cortos.


Para las cuatro métricas, el pvalor es de 0.000. Esto rechaza la hipótesis nula y confirma que existen diferencias significativas según el tamaño del historial.

Aunque los usuarios con historial corto muestran un MAE menor y un NDCG cercano a q por la simplicidad de sus perfiles, los usuarios con un historial de interacciones largo se ven más beneficiados en términos de descubrimiento y utilidad real.

### Hipótesis: El modelo presenta un rendimiento similar en los distintos grupos etarios

In [13]:
from scipy.stats import shapiro

g_joven = df_evaluacion[df_evaluacion['Grupo_Etario'] == "Jóvenes"]
g_adulto = df_evaluacion[df_evaluacion['Grupo_Etario'] == "Adultos"]
g_mayores = df_evaluacion[df_evaluacion['Grupo_Etario'] == "Mayores"]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

for metrica in metricas:
    
    v_joven = g_joven[metrica].dropna()
    v_adulto = g_adulto[metrica].dropna()
    v_mayores = g_mayores[metrica].dropna()

    print(f"\n {metrica}")

    stat, p = shapiro(v_joven)
    print(f"Test de Shapiro-Wilk — jóvenes — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_adulto)
    print(f"Test de Shapiro-Wilk — adultos — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_mayores)
    print(f"Test de Shapiro-Wilk — mayores — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")


 MAE
Test de Shapiro-Wilk — jóvenes — MAE: estadístico=0.880, p-valor=0.000
Test de Shapiro-Wilk — adultos — MAE: estadístico=0.892, p-valor=0.000
Test de Shapiro-Wilk — mayores — MAE: estadístico=0.897, p-valor=0.000

 CG@10
Test de Shapiro-Wilk — jóvenes — CG@10: estadístico=0.623, p-valor=0.000
Test de Shapiro-Wilk — adultos — CG@10: estadístico=0.639, p-valor=0.000
Test de Shapiro-Wilk — mayores — CG@10: estadístico=0.583, p-valor=0.000

 DCG@10
Test de Shapiro-Wilk — jóvenes — DCG@10: estadístico=0.778, p-valor=0.000
Test de Shapiro-Wilk — adultos — DCG@10: estadístico=0.773, p-valor=0.000
Test de Shapiro-Wilk — mayores — DCG@10: estadístico=0.776, p-valor=0.000

 NDCG@10
Test de Shapiro-Wilk — jóvenes — NDCG@10: estadístico=0.385, p-valor=0.000
Test de Shapiro-Wilk — adultos — NDCG@10: estadístico=0.437, p-valor=0.000
Test de Shapiro-Wilk — mayores — NDCG@10: estadístico=0.340, p-valor=0.000


c:\Users\esper\Desktop\tp-recomendacion\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 8091.
  res = hypotest_fun_out(*samples, **kwds)


Ninguna de las métricas evaluadas mediante el test de Shapiro-Wilk presentó un p-valor mayor a 0,05, por lo que se rechaza la hipótesis de normalidad para todas las métricas.

In [14]:
import scipy.stats as stats

for metrica in metricas:
    
    v_joven = g_joven[metrica].dropna()
    v_adulto = g_adulto[metrica].dropna()
    v_mayores = g_mayores[metrica].dropna()
    
    stat, p = stats.levene(v_joven, v_adulto, v_mayores, center='median')
    print(f"Test de Levene para {metrica}: Estadístico={stat:.3f}, p-valor={p:.3f}")

Test de Levene para MAE: Estadístico=1.602, p-valor=0.202
Test de Levene para CG@10: Estadístico=30.748, p-valor=0.000
Test de Levene para DCG@10: Estadístico=30.074, p-valor=0.000
Test de Levene para NDCG@10: Estadístico=10.170, p-valor=0.000


MAE : Cumple el supuesto de homocedasticidad. No hay evidencia estadística de que las varianzas entre los grupos sean distintas.

CG, DCG y NDCG: Violan el supuesto de homocedasticidad. Las varianzas de los grupos son significativamente diferentes enrte sí.

Aplicamos:
* ANOVA para MAE
* Kruskal-Wallis para CG, DCG y NDCG

In [15]:
from scipy import stats

alpha = 0.05

v_joven = g_joven['MAE'].dropna()
v_adulto = g_adulto['MAE'].dropna()
v_mayores = g_mayores['MAE'].dropna()

stat, p = stats.f_oneway(v_joven, v_adulto, v_mayores)
print(f"\nTest de ANOVA — MAE: estadístico={stat:.3f}, p-valor={p:.3f}")
    
if p > alpha:
    print("No hay suficiente evidencia para rechazar la hipótesis nula.")
    print(f"No hay una diferencia significativa en MAE entre los grupos etarios.")
else:
    print("Se rechaza la hipótesis nula.")
    print(f"Existe una diferencia significativa en MAE entre los grupos etarios.")


Test de ANOVA — MAE: estadístico=0.321, p-valor=0.726
No hay suficiente evidencia para rechazar la hipótesis nula.
No hay una diferencia significativa en MAE entre los grupos etarios.


In [16]:
metricas_kw = [f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

for metrica in metricas_kw:
    # Extracción de valores limpios
    v_joven = g_joven[metrica].dropna()
    v_adulto = g_adulto[metrica].dropna()
    v_mayores = g_mayores[metrica].dropna()
    
    #Test de shapiro-wilk
    stat, p = stats.kruskal(v_joven, v_adulto, v_mayores)
    print(f"\nTest de Kruskal-Wallis — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    if p > alpha:
        print("No hay suficiente evidencia para rechazar la hipótesis nula.")
        print(f"No hay una diferencia significativa en {metrica} entre los grupos etarios.")
    else:
        print("Se rechaza la hipótesis nula.")
        print(f"Existe una diferencia significativa en {metrica} entre los grupos etarios.")


Test de Kruskal-Wallis — CG@10: estadístico=22.968, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en CG@10 entre los grupos etarios.

Test de Kruskal-Wallis — DCG@10: estadístico=21.733, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en DCG@10 entre los grupos etarios.

Test de Kruskal-Wallis — NDCG@10: estadístico=35.842, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en NDCG@10 entre los grupos etarios.


La hipótesis nula se mantiene para el error de predicción, lo que indica que el margen de error no cambia según la edad. Sin embargo, se rechaza para CG, DCG y NDCG, confirmando que la cantidad y calidad de las listas recomendadas varían según el grupo etario.

El modelo no presenta un rendimiento completamente homogéneo entre rangos etarios. Presenta un sesgo favorable hacia los adultos y jóvenes en el volumen de contenido relevante.

### Hipótesis: El modelo presenta un rendimiento diferente según el grado de exigencia del usuario.

In [17]:
from scipy.stats import shapiro

#grupo de exigencia: 0 Exigente, 1 normal, 2 generoso
g_exigente = df_evaluacion[df_evaluacion['Grupo_Exigencia'] == 0]
g_normal = df_evaluacion[df_evaluacion['Grupo_Exigencia'] == 1]
g_generoso = df_evaluacion[df_evaluacion['Grupo_Exigencia'] == 2]

metricas = ['MAE', f'CG@{K}', f'DCG@{K}', f'NDCG@{K}']

for metrica in metricas:
    
    v_exigente = g_exigente[metrica].dropna()
    v_normal = g_normal[metrica].dropna()
    v_generoso = g_generoso[metrica].dropna()

    print(f"\n {metrica}")

    stat, p = shapiro(v_exigente)
    print(f"Test de Shapiro-Wilk — exigente — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_normal)
    print(f"Test de Shapiro-Wilk — normal — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    stat, p = shapiro(v_generoso)
    print(f"Test de Shapiro-Wilk — generoso — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")


 MAE
Test de Shapiro-Wilk — exigente — MAE: estadístico=0.945, p-valor=0.000
Test de Shapiro-Wilk — normal — MAE: estadístico=0.881, p-valor=0.000
Test de Shapiro-Wilk — generoso — MAE: estadístico=0.819, p-valor=0.000

 CG@10
Test de Shapiro-Wilk — exigente — CG@10: estadístico=0.507, p-valor=0.000
Test de Shapiro-Wilk — normal — CG@10: estadístico=0.605, p-valor=0.000
Test de Shapiro-Wilk — generoso — CG@10: estadístico=0.446, p-valor=0.000

 DCG@10
Test de Shapiro-Wilk — exigente — DCG@10: estadístico=0.723, p-valor=0.000
Test de Shapiro-Wilk — normal — DCG@10: estadístico=0.734, p-valor=0.000
Test de Shapiro-Wilk — generoso — DCG@10: estadístico=0.535, p-valor=0.000

 NDCG@10
Test de Shapiro-Wilk — exigente — NDCG@10: estadístico=0.264, p-valor=0.000
Test de Shapiro-Wilk — normal — NDCG@10: estadístico=0.429, p-valor=0.000
Test de Shapiro-Wilk — generoso — NDCG@10: estadístico=0.212, p-valor=0.000


c:\Users\esper\Desktop\tp-recomendacion\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 13364.
  res = hypotest_fun_out(*samples, **kwds)


Ninguna de las métricas evaluadas mediante el test de Shapiro-Wilk presentó un p-valor mayor a 0,05, por lo que se rechaza la hipótesis de normalidad para todas las métricas.

In [18]:
import scipy.stats as stats

for metrica in metricas:
    
    v_exigente = g_exigente[metrica].dropna()
    v_normal = g_normal[metrica].dropna()
    v_generoso = g_generoso[metrica].dropna()
    
    stat, p = stats.levene(v_exigente, v_normal, v_generoso, center='median')
    print(f"Test de Levene para {metrica}: Estadístico={stat:.3f}, p-valor={p:.3f}")

Test de Levene para MAE: Estadístico=226.119, p-valor=0.000
Test de Levene para CG@10: Estadístico=99.174, p-valor=0.000
Test de Levene para DCG@10: Estadístico=130.281, p-valor=0.000
Test de Levene para NDCG@10: Estadístico=58.445, p-valor=0.000


Mediante el test de Levene se determinó que las métricas de los grupos separados según la longitud del historial no presentan homocedasticidad.

Dado que no se cumplen los supuestos de normalidad ni de homocedasticidad, se recurre al test no paramétrico de Kruskal-Wallis

In [19]:
alpha = 0.05

for metrica in metricas:
    # Extracción de valores limpios
    v_exigente = g_exigente[metrica].dropna()
    v_normal = g_normal[metrica].dropna()
    v_generoso = g_generoso[metrica].dropna()
    
    #Test de shapiro-wilk
    stat, p = stats.kruskal(v_exigente, v_normal, v_generoso)
    print(f"\nTest de Kruskal-Wallis — {metrica}: estadístico={stat:.3f}, p-valor={p:.3f}")

    if p > alpha:
        print("No hay suficiente evidencia para rechazar la hipótesis nula.")
        print(f"No hay una diferencia significativa en {metrica} entre los grupos de exigencia.")
    else:
        print("Se rechaza la hipótesis nula.")
        print(f"Existe una diferencia significativa en {metrica} entre los grupos de exigencia.")


Test de Kruskal-Wallis — MAE: estadístico=1828.926, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en MAE entre los grupos de exigencia.

Test de Kruskal-Wallis — CG@10: estadístico=2138.431, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en CG@10 entre los grupos de exigencia.

Test de Kruskal-Wallis — DCG@10: estadístico=2388.008, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en DCG@10 entre los grupos de exigencia.

Test de Kruskal-Wallis — NDCG@10: estadístico=242.322, p-valor=0.000
Se rechaza la hipótesis nula.
Existe una diferencia significativa en NDCG@10 entre los grupos de exigencia.


En las cuatro métricas evaluadas, el p-valor es de 0.000. se rechaza la hipótesis nula, lo que confirma que el perfil de exigencia del usuario impacta en el comportamiento del modelo.

El grado de exigencia del usuario condiciona el rendimiento del recomendador. El sisma sufre una penalización crítica ante usuarios exigentes, disparando el error de predicción y reduciendo el contenido relevante entregado.

# Conclusión de KNN ítem-ítem

presenta un raknig muy bueno, con NDCG elevado en todos los grupos. El MAE es similar entre los grupos etarios y entre usuarios con historial corto y largo.

Presenta una diferencia importante según el grado de exigencia: los usuarios exigentes tienen un error de predicción mucho mayor. Esto indeca que el modelo ordena bien los ítems, pero no siempre estima correctamente el valor numérico de sus calificaciones.

# Impacto en modificaciones simples:

## Valores base
* test = 0.2
* KNN = 40
* Similitud = cosine

#### Grupo Etario  
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.364 | 16.817 | 12.113 | 0.988 |
| Jóvenes | 1.353 | 14.232 | 11.141 | 0.991 |
| Mayores | 1.322 | 11.678 |  9.732 | 0.992 |

#### Historial de Interacciones 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Corto | 1.355 |  7.659 |  7.648 | 0.999 |
| Largo | 1.387 | 17.292 | 12.391 | 0.986 |

#### Grupo de Exigencia 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.561 |  7.273 |  6.199 | 0.992 |
| Normal   | 1.207 | 15.626 | 11.602 | 0.979 |
| Generoso | 1.715 | 15.492 | 12.592 | 0.997 |

## Experimento 1: Cambiar tamaño de train/test

#### Grupo Etario (test = 0.1) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.331 | 14.292 | 11.089 | 0.992 |
| Jóvenes | 1.303 | 12.079 | 10.139 | 0.994 |
| Mayores | 1.300 | 10.103 |  8.918 | 0.996 |

#### Historial de Interacciones (test = 0.1)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Corto| 1.345 |  7.652|  7.678 | 0.999 |
|Largo| 1.358 | 14.187| 11.035 | 0.991 |


#### Grupo de Exigencia (test = 0.1)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.456 |  6.955 |  6.073 | 0.995 |
| Normal   | 1.225 | 13.334 | 10.626 | 0.992 |
| Generoso | 1.568 | 14.208 | 11.961 | 0.997 |

#### Grupo Etario (test = 0.3) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.378 | 18.390 | 12.759 | 0.986 |
| Jóvenes | 1.373 | 15.922 | 11.850 | 0.989 |
| Mayores | 1.348 | 13.118 | 10.368 | 0.990 |

#### Historial de Interacciones (test = 0.3)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Corto | 1.355 |  7.669 |  7.655 | 0.999 |
| Largo | 1.407 | 19.511 | 13.358 | 0.983 |


#### Grupo de Exigencia (test = 0.3)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.649 |  7.649 |  6.366 | 0.989 |
| Normal   | 1.192 | 17.111 | 12.231 | 0.987 |
| Generoso | 1.797 | 16.233 | 12.917 | 0.996 |

Efectos detectados:
* A medida que disminuyen los datos de entrenamiento se dregrada la precisión (MAE). Un conjunto de entrenemiento más amplio ayuda a ajustar mejor las estimaciones puntuales del algoritmo.
* Al ampliar el tamaño de la muestra de prueba, hay un volumen mayor de ítems verdaderos disponibles en la evaluación para ser evaluados en el top.
* Los usuarios con historial corto mantienen prácticamente congelado el CG en las tres particiones.

## Experimento 2: Variar K en KNN

#### Grupo Etario (K = 20) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.364 | 16.813 | 12.112 | 0.988 |
| Jóvenes | 1.353 | 14.231 | 11.141 | 0.991 |
| Mayores | 1.322 | 11.678 |  9.732 | 0.992 |

#### Historial de Interacciones (K = 20)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Corto| 1.355 |  7.659 |  7.648 | 0.999 |
|Largo| 1.387 | 17.288 | 12.392 | 0.986 |


#### Grupo de Exigencia (K = 20)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.561 |  7.269 |  6.198 | 0.992 |
| Normal   | 1.207 | 15.623 | 11.601 | 0.989 |
| Generoso | 1.716 | 15.492 | 12.592 | 0.997 |

#### Grupo Etario (K = 60) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.364 | 16.819 | 12.114 | 0.988 |
| Jóvenes | 1.353 | 14.233 | 11.141 | 0.991 |
| Mayores | 1.322 | 11.678 |  9.732 | 0.992 |

#### Historial de Interacciones (K = 60)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Corto | 1.355 |  7.659 |  7.648 | 0.999 |
| Largo | 1.387 | 17.293 | 12.393 | 0.986 |


#### Grupo de Exigencia (K = 60)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.561 |  7.273 |  6.198 | 0.992 |
| Normal   | 1.207 | 15.627 | 11.602 | 0.989 |
| Generoso | 1.715 | 15.492 | 12.592 | 0.997 |

Efectos detectados:
* Comportamiento similar en KNN usuario - usuario.
* Los vecinos adicionales no aportan información nueva debido a la dispersión de los datos o a que su similitud es demasiado baja para alterar la predicción.
* Conviene mantener K = 40 o probar valores menores para optimizar recursos computacionales.

## Experimento 3: variar similitud

#### Grupo Etario (pearson) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.355 | 16.826 | 12.122 | 0.988 |
| Jóvenes | 1.362 | 14.233 | 11.144 | 0.991 |
| Mayores | 1.348 | 11.665 |  9.732 | 0.993 |

#### Historial de Interacciones (pearson)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
|Corto| 1.354 |  7.659 |  7.648 | 0.999 |
|Largo| 1.377 | 17.295 | 12.401 | 0.987 |


#### Grupo de Exigencia (pearson)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.657 |  7.267 |  6.196 | 0.992 |
| Normal   | 1.172 | 15.629 | 11.609 | 0.989 |
| Generoso | 1.853 | 15.495 | 12.598 | 0.997 |

#### Grupo Etario (msd) 
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Adultos | 1.371 | 16.822 | 12.112 | 0.988 |
| Jóvenes | 1.362 | 14.228 | 11.134 | 0.991 |
| Mayores | 1.329 | 11.675 |  9.725 | 0.992 |

#### Historial de Interacciones (msd)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Corto | 1.355 |  7.659 |  7.648 | 0.999 |
| Largo | 1.395 | 17.294 | 12.388 | 0.986 |


#### Grupo de Exigencia (msd)
|GRUPO|MAE|CG|DCG|NDCG|
|---|---:|---:|---:|---:|
| Exigente | 2.565 |  7.271 |  6.194 | 0.992 |
| Normal   | 1.214 | 15.628 | 11.598 | 0.989 |
| Generoso | 1.718 | 15.498 | 12.592 | 0.997 |

Efectos detectados:
* El cambio de la función de similitud no altera el orden ni la relevancia de las recomendaciones de las listas Top-10, aunque genera redistribuciones en la precisión (MAE).
* Pearson
    * Mejora el mae de los usuarios promedio.
    * Degrada el mae de los usuarios exigentes y generosos.
* MSD es prácticamente equivalente al coseno.

* El cambio de la métrica de similitud produce un impacto prácticamente nulo en el ranking de recomendaciones.
* Pearson aporta una leve mejora en la precisión (MAE).
* Las diferencias entre los grupos se mantiene intactas. 
* MSD y Coseno presentan un rendimiento practicamente idénticos.


